# People Playground Unity source export

This notebook clones the Unity repository, inventories `.meta` GUIDs and references, extracts prefab and scene structures, indexes C# systems and managed binaries, copies `.byte`/`.bytes` payloads, and generates GUID-keyed pixel modules for images under `Assets/Texture2D`.

## Credential safety

The GitHub token previously pasted into chat is exposed. Revoke it in GitHub settings before continuing. Do not put a token in this notebook or a committed file. If the repository is private, add a newly created fine-grained, read-only token as a Colab Secret named `PPG_GITHUB_TOKEN`; the code reads it from Colab Secrets and never prints or saves it. Public repositories need no token.

The export is a review bundle, not an automatic C#-to-Luau translator or a Roblox-ready full-fidelity asset upload. Check `SUMMARY.md` and all error/unsupported lists before using the data.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/johnesecat/PPG-UnityProject-Files-Duplicate.git"
REPO_DIR = Path("/content/PPG-UnityProject-Files-Duplicate")

if not REPO_DIR.exists():
    env = os.environ.copy()
    askpass = Path("/tmp/ppg-git-askpass.sh")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("PPG_GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        askpass.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s\\n' x-access-token ;; *) printf '%s\\n' \"$PPG_GITHUB_TOKEN\" ;; esac\n", encoding="utf-8")
        askpass.chmod(0o700)
        env["PPG_GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], env=env, text=True, capture_output=True)
    if askpass.exists():
        askpass.unlink()
    env.pop("PPG_GITHUB_TOKEN", None)
    token = None
    if result.returncode != 0:
        raise RuntimeError("Clone failed. For a private repo, add a fresh token as the PPG_GITHUB_TOKEN Colab Secret after revoking the exposed token.\n" + result.stderr[-2000:])

ASSET_ROOT = REPO_DIR / "ppg/People Playground/Assets"
assert ASSET_ROOT.is_dir(), f"Assets folder not found: {ASSET_ROOT}"
print("Source checkout ready; credentials were not written to disk.")

In [ ]:
import sys
try:
    import PIL
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Pillow"], check=True)

OUTPUT_DIR = Path("/content/ppg-source-export")
COPY_ORIGINAL_TEXTURES = False  # True includes original image files and can make a very large ZIP.
command = [
    sys.executable, str(REPO_DIR / "tools/colab_source_export.py"),
    "--assets", str(ASSET_ROOT),
    "--output", str(OUTPUT_DIR),
    "--max-image-dimension", "96",
    "--max-source-pixels", "16777216",
    "--max-file-mb", "64",
    "--scene-limit", "8",
    "--zip",
]
if COPY_ORIGINAL_TEXTURES:
    command.append("--copy-textures")
subprocess.run(command, check=True)
print(f"Export directory: {OUTPUT_DIR}")

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT_DIR / "SUMMARY.md").read_text(encoding="utf-8")))
print("Bundle size (MB):", round(OUTPUT_DIR.with_suffix(".zip").stat().st_size / (1024 * 1024), 2))

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR.with_suffix(".zip")))